# Optimal Traveler Strategy Tool

## Setup & Data Loading

In [115]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_excel('airline_ticket_dataset.xlsx')

# Create derived features
df['route'] = df['city1'] + ' → ' + df['city2']
df['price_per_mile'] = df['fare'] / df['nsmiles']
df['city_pair'] = df.apply(lambda row: ' ↔ '.join(sorted([row['city1'], row['city2']])), axis=1)

print("Data loaded successfully!")

Data loaded successfully!


## Available Cities in Dataset

Let's see what cities are available so you know what to input!

In [116]:
# Get all unique cities
all_cities = sorted(list(set(df['city1'].unique()) | set(df['city2'].unique())))

print(f"Dataset contains {len(all_cities)} cities\n")
print("=" * 80)
print("AVAILABLE CITIES (copy/paste the exact name when inputting):")
print("=" * 80)

# Display in columns for easier reading
for i, city in enumerate(all_cities, 1):
    print(f"{i:3d}. {city}")
    
print("\n" + "=" * 80)
print("Note: Be sure to copy the exact city name (including state/area designation)")
print("=" * 80)

Dataset contains 136 cities

AVAILABLE CITIES (copy/paste the exact name when inputting):
  1. Albany, NY
  2. Albuquerque, NM
  3. Allentown/Bethlehem/Easton, PA
  4. Amarillo, TX
  5. Appleton, WI
  6. Asheville, NC
  7. Aspen, CO
  8. Atlanta, GA (Metropolitan Area)
  9. Atlantic City, NJ
 10. Austin, TX
 11. Bangor, ME
 12. Belleville, IL
 13. Bellingham, WA
 14. Bend/Redmond, OR
 15. Billings, MT
 16. Birmingham, AL
 17. Bismarck/Mandan, ND
 18. Boise, ID
 19. Boston, MA (Metropolitan Area)
 20. Bozeman, MT
 21. Buffalo, NY
 22. Burlington, VT
 23. Cedar Rapids/Iowa City, IA
 24. Charleston, SC
 25. Charlotte, NC
 26. Charlottesville, VA
 27. Chicago, IL
 28. Cincinnati, OH
 29. Cleveland, OH (Metropolitan Area)
 30. Colorado Springs, CO
 31. Columbia, SC
 32. Columbus, OH
 33. Dallas/Fort Worth, TX
 34. Dayton, OH
 35. Denver, CO
 36. Des Moines, IA
 37. Detroit, MI
 38. Eagle, CO
 39. El Paso, TX
 40. Eugene, OR
 41. Eureka/Arcata, CA
 42. Everett, WA
 43. Fargo, ND
 44. Fayette

In [117]:
def calculate_route_scores(df_input):
    """
    Calculate affordability scores for each route.
    Higher score = better deal for travelers.
    """
    df_scored = df_input.copy()
    
    # 1. Competition Score (0-25 points): Based on LCC market share
    df_scored['competition_score'] = (df_scored['lf_ms'] / df_scored['lf_ms'].max()) * 25
    
    # 2. Efficiency Score (0-25 points): Based on price per mile (inverted)
    max_ppm = df_scored['price_per_mile'].quantile(0.95)  # Use 95th percentile to avoid outliers
    df_scored['efficiency_score'] = (1 - (df_scored['price_per_mile'] / max_ppm)) * 25
    df_scored['efficiency_score'] = df_scored['efficiency_score'].clip(0, 25)
    
    # 3. Market Power Score (0-25 points): Lower carrier dominance = better
    df_scored['market_score'] = (1 - df_scored['large_ms']) * 25
    
    # 4. Price Score (0-25 points): Lower absolute fare = better
    max_fare = df_scored['fare'].quantile(0.95)
    df_scored['price_score'] = (1 - (df_scored['fare'] / max_fare)) * 25
    df_scored['price_score'] = df_scored['price_score'].clip(0, 25)
    
    # Total Affordability Score (0-100)
    df_scored['affordability_score'] = (
        df_scored['competition_score'] + 
        df_scored['efficiency_score'] + 
        df_scored['market_score'] + 
        df_scored['price_score']
    )
    
    return df_scored

# Apply scoring
df_scored = calculate_route_scores(df)

print("Route scoring complete!")
print(f"\n Score Distribution:")
print(f"   Mean: {df_scored['affordability_score'].mean():.1f}")
print(f"   Median: {df_scored['affordability_score'].median():.1f}")
print(f"   Best possible route: {df_scored['affordability_score'].max():.1f}")
print(f"   Worst route: {df_scored['affordability_score'].min():.1f}")

Route scoring complete!

 Score Distribution:
   Mean: 41.2
   Median: 42.7
   Best possible route: 65.9
   Worst route: 2.1


## Part 1: Route Affordability Scoring System

We'll create a **composite score (0-100)** for each route based on:
- **Competition Score**: Higher LCC presence = better
- **Efficiency Score**: Lower price per mile = better
- **Market Power Score**: Lower carrier dominance = better
- **Price Score**: Lower absolute fare = better

In [118]:
# Get best routes
best_routes = df_scored.groupby('route').agg({
    'affordability_score': 'mean',
    'fare': 'mean',
    'price_per_mile': 'mean',
    'lf_ms': 'mean',
    'nsmiles': 'mean',
    'passengers': 'sum'
}).reset_index().nlargest(30, 'affordability_score')

fig = px.bar(best_routes, 
             x='affordability_score', 
             y='route',
             orientation='h',
             title='Top 30 Best Value Routes (Affordability Score)',
             labels={'affordability_score': 'Affordability Score (0-100)', 'route': 'Route'},
             color='fare',
             color_continuous_scale='RdYlGn_r',
             hover_data={'fare': ':.2f', 'lf_ms': ':.1%', 'price_per_mile': ':.3f', 'nsmiles': ':.0f'})

fig.update_layout(height=900, yaxis={'categoryorder':'total ascending'})
fig.show()

print("\n These routes offer the best combination of:")
print("   ✓ Strong competition (high LCC presence)")
print("   ✓ Low market concentration")
print("   ✓ Efficient pricing (good $/mile)")
print("   ✓ Affordable absolute fares")


 These routes offer the best combination of:
   ✓ Strong competition (high LCC presence)
   ✓ Low market concentration
   ✓ Efficient pricing (good $/mile)
   ✓ Affordable absolute fares


### Top 30 Best Value Routes by Affordability Score

Here are the overall best routes in the dataset - use these for inspiration!

In [119]:
# Analyze score components for top routes
top_route = best_routes.iloc[0]['route']
top_route_data = df_scored[df_scored['route'] == top_route].iloc[0]

components = pd.DataFrame({
    'Component': ['Competition\n(LCC Presence)', 'Efficiency\n($/mile)', 'Market Power\n(Low Dominance)', 'Price\n(Absolute Fare)'],
    'Score': [top_route_data['competition_score'], 
              top_route_data['efficiency_score'],
              top_route_data['market_score'],
              top_route_data['price_score']],
    'Max': [25, 25, 25, 25]
})

fig = go.Figure()
fig.add_trace(go.Bar(name='Score', x=components['Component'], y=components['Score'], 
                     text=components['Score'], texttemplate='%{text:.1f}/25',
                     marker_color='lightseagreen'))
fig.add_trace(go.Bar(name='Remaining', x=components['Component'], 
                     y=components['Max'] - components['Score'],
                     marker_color='lightgray'))

fig.update_layout(barmode='stack', 
                  title=f'Score Breakdown: {top_route} (Best Route in Dataset)',
                  yaxis_title='Score (out of 25)',
                  height=500,
                  showlegend=False)
fig.show()

print(f"\n Best Route: {top_route}")
print(f"   Total Score: {top_route_data['affordability_score']:.1f}/100")
print(f"   Average Fare: ${top_route_data['fare']:.2f}")
print(f"   LCC Market Share: {top_route_data['lf_ms']:.1%}")
print(f"   Price per Mile: ${top_route_data['price_per_mile']:.3f}")


 Best Route: Tampa, FL (Metropolitan Area) → Trenton, NJ
   Total Score: 63.9/100
   Average Fare: $98.77
   LCC Market Share: 100.0%
   Price per Mile: $0.103


**What each component measures:**

1. **Competition (LCC Presence)**: How many low-cost carriers (like Spirit, Frontier) fly this route. More budget airlines = more competition = better deals for you.

2. **Efficiency ($/mile)**: Cost per mile traveled. A shorter, expensive flight might cost more per mile than a long, cheaper flight. Lower cost per mile = better value.

3. **Market Power (Low Dominance)**: Whether one big airline controls the route. If United has 90% of flights on a route, they can charge whatever they want. Less dominance = better prices.

4. **Price (Absolute Fare)**: The actual ticket price. Lower fare = better score.

### Score Components Breakdown

This shows how the #1 best route in the market achieves its high affordability score.

---

## USER INPUT SECTION - CUSTOMIZE YOUR ANALYSIS HERE!

In [120]:
# ============================================================================
# EDIT THESE VALUES TO ANALYZE YOUR OWN ROUTES
# ============================================================================

# YOUR TRAVEL DETAILS (edit these!)
my_origin = "Los Angeles, CA (Metropolitan Area)"           # [CHANGE] THIS to your departure city
my_destination = "New York City, NY (Metropolitan Area)"        # [CHANGE] THIS to your arrival city

# TRAVEL TIMING (optional - leave as None for recommendations)
departure_quarter = None      # [OPTIONAL] Set to 1, 2, 3, or 4 (or None for recommendation)
return_quarter = None         # [OPTIONAL] Set to 1, 2, 3, or 4 (or None for recommendation)

# ============================================================================
# Validation and display
# ============================================================================

print("YOUR TRAVEL PROFILE:")
print("=" * 70)
print(f"Origin: {my_origin}")
print(f"Destination: {my_destination}")
print("=" * 70)

# Check if cities exist in dataset
origin_exists = my_origin in all_cities
dest_exists = my_destination in all_cities

if not origin_exists:
    print(f"\n WARNING: '{my_origin}' not found in dataset!")
    print("   Please copy a city name from the list above.")
    
if not dest_exists:
    print(f"\n WARNING: '{my_destination}' not found in dataset!")
    print("   Please copy a city name from the list above.")

if origin_exists and dest_exists:
    print("\nBoth cities found! Ready to analyze your route.")
    
print("\n To change your inputs, edit the variables in this cell and re-run it.")

YOUR TRAVEL PROFILE:
Origin: Los Angeles, CA (Metropolitan Area)
Destination: New York City, NY (Metropolitan Area)

Both cities found! Ready to analyze your route.

 To change your inputs, edit the variables in this cell and re-run it.


In [121]:
# Analyze score components for YOUR route
your_route = f"{my_origin} → {my_destination}"
your_route_data = df_scored[df_scored['route'] == your_route]

if len(your_route_data) > 0:
    # Get average scores for your route
    your_route_avg = your_route_data.iloc[0]
    
    components = pd.DataFrame({
        'Component': ['Competition\n(LCC Presence)', 'Efficiency\n($/mile)', 'Market Power\n(Low Dominance)', 'Price\n(Absolute Fare)'],
        'Score': [your_route_avg['competition_score'], 
                  your_route_avg['efficiency_score'],
                  your_route_avg['market_score'],
                  your_route_avg['price_score']],
        'Max': [25, 25, 25, 25]
    })
    
    fig = go.Figure()
    fig.add_trace(go.Bar(name='Score', x=components['Component'], y=components['Score'], 
                         text=components['Score'], texttemplate='%{text:.1f}/25',
                         marker_color='mediumpurple'))
    fig.add_trace(go.Bar(name='Remaining', x=components['Component'], 
                         y=components['Max'] - components['Score'],
                         marker_color='lightgray'))
    
    fig.update_layout(barmode='stack', 
                      title=f'Score Breakdown: {your_route} (Your Route)',
                      yaxis_title='Score (out of 25)',
                      height=500,
                      showlegend=False)
    fig.show()
    
    print(f"\n YOUR ROUTE: {your_route}")
    print(f"   Total Score: {your_route_avg['affordability_score']:.1f}/100")
    print(f"   Average Fare: ${your_route_avg['fare']:.2f}")
    print(f"   LCC Market Share: {your_route_avg['lf_ms']:.1%}")
    print(f"   Price per Mile: ${your_route_avg['price_per_mile']:.3f}")
    
    # Compare to dataset's best route
    best_score = df_scored['affordability_score'].max()
    score_diff = your_route_avg['affordability_score'] - best_score
    
    print(f"\n COMPARISON TO BEST ROUTE:")
    if score_diff >= 0:
        print(f"   Your route IS the best route in the dataset!")
    elif score_diff > -10:
        print(f"   Your route is excellent! Only {abs(score_diff):.1f} points below the best.")
    elif score_diff > -20:
        print(f"   Your route is good! {abs(score_diff):.1f} points below the best.")
    else:
        print(f"   Your route scores {abs(score_diff):.1f} points below the best. Check alternatives below!")
else:
    print(f"\n Route '{your_route}' not found in dataset.")
    print("   Please check your origin and destination cities above.")



 YOUR ROUTE: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)
   Total Score: 42.5/100
   Average Fare: $430.38
   LCC Market Share: 22.7%
   Price per Mile: $0.171

 COMPARISON TO BEST ROUTE:
   Your route scores 23.5 points below the best. Check alternatives below!


### Your Route's Score Breakdown

Let's analyze how YOUR specific route performs on each scoring component.

## Part 2: Alternative Route Finder

Find cheaper alternatives by considering nearby airports or different city pairs.

In [122]:
def find_alternative_routes(origin, destination, df_data, max_alternatives=10, preferred_quarter=None):
    """
    Find alternative routes and nearby airports that might offer better deals.
    If preferred_quarter is specified, prioritize routes in that quarter.
    """
    # Direct route
    direct_route = f"{origin} → {destination}"
    direct_data = df_data[df_data['route'] == direct_route]
    
    if len(direct_data) == 0:
        # Try reverse
        direct_route = f"{destination} → {origin}"
        direct_data = df_data[df_data['route'] == direct_route]
    
    if len(direct_data) == 0:
        return None, pd.DataFrame()
    
    direct_avg = direct_data.groupby('route').agg({
        'fare': 'mean',
        'affordability_score': 'mean',
        'lf_ms': 'mean',
        'large_ms': 'mean',
        'nsmiles': 'mean'
    }).reset_index()
    
    # Find alternative routes departing from the same origin
    base_filter = (df_data['city1'] == origin) & (df_data['route'] != direct_route)
    
    if preferred_quarter is not None:
        # Try to find alternatives in the same quarter first
        alternative_routes = df_data[base_filter & (df_data['quarter'] == preferred_quarter)].copy()
        
        # If not enough alternatives, expand to adjacent quarters
        if len(alternative_routes) < max_alternatives:
            # Calculate quarter distances (circular: Q1 is close to Q4)
            def quarter_distance(q1, q2):
                diff = abs(q1 - q2)
                return min(diff, 4 - diff)
            
            # Add routes from other quarters, sorted by quarter proximity
            other_quarters = df_data[base_filter & (df_data['quarter'] != preferred_quarter)].copy()
            other_quarters['quarter_distance'] = other_quarters['quarter'].apply(lambda q: quarter_distance(preferred_quarter, q))
            other_quarters = other_quarters.sort_values('quarter_distance')
            alternative_routes = pd.concat([alternative_routes, other_quarters])
    else:
        alternative_routes = df_data[base_filter].copy()
    
    alternatives = alternative_routes.groupby('route').agg({
        'fare': 'mean',
        'affordability_score': 'mean',
        'lf_ms': 'mean',
        'large_ms': 'mean',
        'nsmiles': 'mean',
        'city1': 'first',
        'city2': 'first',
        'quarter': lambda x: ', '.join(sorted(set(f'Q{int(q)}' for q in x)))
    }).reset_index()
    
    # Calculate potential savings
    if len(direct_avg) > 0:
        direct_fare = direct_avg.iloc[0]['fare']
        alternatives['savings'] = direct_fare - alternatives['fare']
        alternatives['savings_pct'] = (alternatives['savings'] / direct_fare) * 100
        
        # Filter to only cheaper alternatives
        alternatives = alternatives[alternatives['savings'] > 0].nlargest(max_alternatives, 'savings')
    
    return direct_avg.iloc[0] if len(direct_avg) > 0 else None, alternatives

# Use the user's input cities
print(f"Analyzing YOUR route: {my_origin} → {my_destination}")
print("="*80)

Analyzing YOUR route: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)


In [123]:
# Run alternative route analysis for YOUR cities
direct, alternatives = find_alternative_routes(my_origin, my_destination, df_scored, preferred_quarter=departure_quarter)

if direct is not None:
    print(f"\n DIRECT ROUTE: {direct['route']}")
    print(f"   Average Fare: ${direct['fare']:.2f}")
    print(f"   Affordability Score: {direct['affordability_score']:.1f}/100")
    print(f"   Distance: {direct['nsmiles']:.0f} miles")
    print(f"   LCC Presence: {direct['lf_ms']:.1%}")
    
    if len(alternatives) > 0:
        print(f"\n\n FOUND {len(alternatives)} CHEAPER ALTERNATIVES:")
        print("="*80)
        
        for idx, alt in alternatives.head(5).iterrows():
            print(f"\n{idx+1}. {alt['route']}")
            print(f"   Save: ${alt['savings']:.2f} ({alt['savings_pct']:.1f}%)")
            print(f"   Fare: ${alt['fare']:.2f} | Score: {alt['affordability_score']:.1f}/100")
            print(f"   Distance: {alt['nsmiles']:.0f} miles | LCC: {alt['lf_ms']:.1%}")
            
            # Show available quarters
            quarter_info = f"   Quarters: {alt['quarter']}"
            if departure_quarter is not None:
                if f'Q{departure_quarter}' in alt['quarter']:
                    quarter_info += " ✓"
            print(quarter_info)
            
            # Suggest strategy
            other_city = alt['city2']
            print(f"   💡 Strategy: Fly from {my_origin} to {other_city} instead")
    else:
        print("\n This is already one of the best options available!")
else:
    print("Route not found in dataset. Try another city pair.")


 DIRECT ROUTE: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)
   Average Fare: $412.23
   Affordability Score: 43.2/100
   Distance: 2510 miles
   LCC Presence: 25.7%


 FOUND 10 CHEAPER ALTERNATIVES:

19. Los Angeles, CA (Metropolitan Area) → Provo, UT
   Save: $318.31 (77.2%)
   Fare: $93.92 | Score: 52.5/100
   Distance: 568 miles | LCC: 31.3%
   Quarters: Q1, Q2, Q3, Q4
   💡 Strategy: Fly from Los Angeles, CA (Metropolitan Area) to Provo, UT instead

26. Los Angeles, CA (Metropolitan Area) → San Francisco, CA (Metropolitan Area)
   Save: $259.71 (63.0%)
   Fare: $152.52 | Score: 42.6/100
   Distance: 372 miles | LCC: 36.8%
   Quarters: Q1, Q2, Q3, Q4
   💡 Strategy: Fly from Los Angeles, CA (Metropolitan Area) to San Francisco, CA (Metropolitan Area) instead

21. Los Angeles, CA (Metropolitan Area) → Reno, NV
   Save: $259.61 (63.0%)
   Fare: $152.62 | Score: 43.0/100
   Distance: 415 miles | LCC: 38.7%
   Quarters: Q1, Q2, Q3, Q4
   💡 Strategy: Fly fro

## Part 3: Optimal Booking Window

Identify the best quarters to travel based on historical patterns.

In [124]:
# Filter data for user's specific route
user_route = df_scored[
    ((df_scored['city1'] == my_origin) & (df_scored['city2'] == my_destination))
].copy()

if len(user_route) == 0:
    print(f"No data found for route {my_origin} → {my_destination}")
    print("Please check your origin/destination cities.")
else:
    # Analyze quarterly patterns for this specific route
    route_quarterly = user_route.groupby('quarter').agg({
        'fare': ['mean', 'min', 'max'],
        'affordability_score': 'mean',
        'passengers': 'sum'
    }).reset_index()
    
    route_quarterly.columns = ['Quarter', 'Mean Fare', 'Min Fare', 'Max Fare', 'Avg Score', 'Total Passengers']
    
    quarter_labels = {1: 'Q1\n(Jan-Mar)', 2: 'Q2\n(Apr-Jun)', 3: 'Q3\n(Jul-Sep)', 4: 'Q4\n(Oct-Dec)'}
    route_quarterly['Quarter Label'] = route_quarterly['Quarter'].map(quarter_labels)
    
    # ========================================================================
    # CONDITIONAL LOGIC: Recommendations vs. Cost Estimates
    # ========================================================================
    
    if departure_quarter is None and return_quarter is None:
        # USER DID NOT SPECIFY QUARTERS - PROVIDE RECOMMENDATIONS
        print("\n" + "="*70)
        print(" PERSONALIZED QUARTER RECOMMENDATIONS")
        print(f"    Route: {my_origin} → {my_destination}")
        print("="*70)
        
        best_q = route_quarterly.loc[route_quarterly['Mean Fare'].idxmin()]
        worst_q = route_quarterly.loc[route_quarterly['Mean Fare'].idxmax()]
        savings_potential = worst_q['Mean Fare'] - best_q['Mean Fare']
        
        print(f"\n✅ BEST QUARTER TO TRAVEL: {quarter_labels[best_q['Quarter']].replace(chr(10), ' ')}")
        print(f"   Expected Fare: ${best_q['Mean Fare']:.2f}")
        print(f"   Fare Range: ${best_q['Min Fare']:.2f} - ${best_q['Max Fare']:.2f}")
        print(f"   Affordability Score: {best_q['Avg Score']:.1f}/100")
        
        print(f"\n❌ AVOID: {quarter_labels[worst_q['Quarter']].replace(chr(10), ' ')}")
        print(f"   Expected Fare: ${worst_q['Mean Fare']:.2f}")
        print(f"   💰 Potential Savings: ${savings_potential:.2f} ({savings_potential/worst_q['Mean Fare']*100:.1f}% cheaper)")
        
        print("\n📊 ALL QUARTERS FOR YOUR ROUTE:")
        for _, row in route_quarterly.sort_values('Mean Fare').iterrows():
            marker = "✅" if row['Quarter'] == best_q['Quarter'] else "❌" if row['Quarter'] == worst_q['Quarter'] else "  "
            print(f"   {marker} {quarter_labels[row['Quarter']].replace(chr(10), ' '):20} ${row['Mean Fare']:7.2f}  (Score: {row['Avg Score']:.1f}/100)")
    
    else:
        # USER SPECIFIED QUARTERS - SHOW COST ESTIMATES
        print("\n" + "="*70)
        print(" 💰 YOUR TRAVEL COST ESTIMATE")
        print(f"    Route: {my_origin} → {my_destination}")
        print("="*70)
        
        total_cost = 0
        
        if departure_quarter is not None:
            dep_data = route_quarterly[route_quarterly['Quarter'] == departure_quarter]
            if len(dep_data) > 0:
                dep_fare = dep_data.iloc[0]['Mean Fare']
                dep_score = dep_data.iloc[0]['Avg Score']
                dep_range_low = dep_data.iloc[0]['Min Fare']
                dep_range_high = dep_data.iloc[0]['Max Fare']
                total_cost += dep_fare
                
                print(f"\n✈️  DEPARTURE: {quarter_labels[departure_quarter].replace(chr(10), ' ')}")
                print(f"   Expected Fare: ${dep_fare:.2f}")
                print(f"   Typical Range: ${dep_range_low:.2f} - ${dep_range_high:.2f}")
                print(f"   Affordability: {dep_score:.1f}/100")
            else:
                print(f"\n⚠️  No data for departure in Q{departure_quarter}")
        
        if return_quarter is not None:
            ret_data = route_quarterly[route_quarterly['Quarter'] == return_quarter]
            if len(ret_data) > 0:
                ret_fare = ret_data.iloc[0]['Mean Fare']
                ret_score = ret_data.iloc[0]['Avg Score']
                ret_range_low = ret_data.iloc[0]['Min Fare']
                ret_range_high = ret_data.iloc[0]['Max Fare']
                total_cost += ret_fare
                
                print(f"\n🏠 RETURN: {quarter_labels[return_quarter].replace(chr(10), ' ')}")
                print(f"   Expected Fare: ${ret_fare:.2f}")
                print(f"   Typical Range: ${ret_range_low:.2f} - ${ret_range_high:.2f}")
                print(f"   Affordability: {ret_score:.1f}/100")
            else:
                print(f"\n⚠️  No data for return in Q{return_quarter}")
        
        if total_cost > 0:
            print(f"\n💵 TOTAL ESTIMATED COST: ${total_cost:.2f}")
            
            # Compare to best option
            best_q = route_quarterly.loc[route_quarterly['Mean Fare'].idxmin()]
            if departure_quarter == best_q['Quarter'] or return_quarter == best_q['Quarter']:
                print(f"   ✅ You're traveling in the cheapest quarter!")
            else:
                potential_savings = total_cost - (best_q['Mean Fare'] * 2 if return_quarter is not None and departure_quarter is not None else best_q['Mean Fare'])
                if potential_savings > 0:
                    print(f"   💡 TIP: Traveling in {quarter_labels[best_q['Quarter']].replace(chr(10), ' ')} could save ~${potential_savings:.2f}")
    
    # Visualization
    fig = make_subplots(rows=1, cols=2, 
                        subplot_titles=(f'Average Fare by Quarter<br>{my_origin} → {my_destination}', 
                                       'Affordability Score by Quarter'),
                        specs=[[{"type": "bar"}, {"type": "bar"}]])
    
    # Highlight user's selected quarters
    if departure_quarter is not None or return_quarter is not None:
        colors = []
        for q in route_quarterly['Quarter']:
            if q in [departure_quarter, return_quarter]:
                colors.append('#9b59b6')  # Purple for selected
            elif q == route_quarterly.loc[route_quarterly['Mean Fare'].idxmin(), 'Quarter']:
                colors.append('#2ecc71')  # Green for cheapest
            else:
                colors.append('#3498db')  # Blue for others
    else:
        colors = ['#2ecc71' if x == route_quarterly['Mean Fare'].min() else 
                 '#e74c3c' if x == route_quarterly['Mean Fare'].max() else 
                 '#3498db' for x in route_quarterly['Mean Fare']]
    
    fig.add_trace(
        go.Bar(x=route_quarterly['Quarter Label'], y=route_quarterly['Mean Fare'],
               text=route_quarterly['Mean Fare'], texttemplate='$%{text:.2f}',
               marker_color=colors,
               name='Mean Fare'),
        row=1, col=1
    )
    
    score_colors = ['#2ecc71' if x == route_quarterly['Avg Score'].max() else 
                   '#e74c3c' if x == route_quarterly['Avg Score'].min() else 
                   '#3498db' for x in route_quarterly['Avg Score']]
    
    fig.add_trace(
        go.Bar(x=route_quarterly['Quarter Label'], y=route_quarterly['Avg Score'],
               text=route_quarterly['Avg Score'], texttemplate='%{text:.1f}',
               marker_color=score_colors,
               name='Avg Score'),
        row=1, col=2
    )
    
    fig.update_xaxes(title_text="Quarter", row=1, col=1)
    fig.update_xaxes(title_text="Quarter", row=1, col=2)
    fig.update_yaxes(title_text="Average Fare ($)", row=1, col=1)
    fig.update_yaxes(title_text="Affordability Score", row=1, col=2)
    
    fig.update_layout(height=500, showlegend=False, 
                     title_text=f"Optimal Travel Timing Analysis<br><sub>Your Route: {my_origin} ↔ {my_destination}</sub>")
    fig.show()



 PERSONALIZED QUARTER RECOMMENDATIONS
    Route: Los Angeles, CA (Metropolitan Area) → New York City, NY (Metropolitan Area)

✅ BEST QUARTER TO TRAVEL: Q1 (Jan-Mar)
   Expected Fare: $378.21
   Fare Range: $315.77 - $415.08
   Affordability Score: 43.8/100

❌ AVOID: Q4 (Oct-Dec)
   Expected Fare: $441.21
   💰 Potential Savings: $63.00 (14.3% cheaper)

📊 ALL QUARTERS FOR YOUR ROUTE:
   ✅ Q1 (Jan-Mar)         $ 378.21  (Score: 43.8/100)
      Q3 (Jul-Sep)         $ 401.46  (Score: 43.4/100)
      Q2 (Apr-Jun)         $ 432.60  (Score: 42.9/100)
   ❌ Q4 (Oct-Dec)         $ 441.21  (Score: 42.4/100)
